# TGEN-936CI - Gubbins Results Processing

### Date: 25/07/21


# Import statements

In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm

import matplotlib.pyplot as plt
import seaborn as sns
#import pickle

%matplotlib inline

In [2]:
# https://bioframe.readthedocs.io/en/latest/guide-intervalops.html
import bioframe as bf


In [3]:
import json

import ete3 as ETE

from ete3 import Tree


### Import custom functions

In [4]:
from gcutils.gubbinsfuncs import get_RecombEvents_From_Gubbins_GFF

from gcutils.gubbinsfuncs import parse_BaseReconstruction_EMBL_Gubbins, annotate_Gubbins_SNP_Events_By_RecombEventID

from gcutils.gubbinsfuncs import get_RecombEvents_From_Gubbins_GFF_V2, label_Gubbins_Events_DF_ByOvrLap_H37RvGenes, label_Gubbins_Events_DF_ByOvrLap_HHRs

from gcutils.gubbinsfuncs import get_BranchLengths_and_DescendantCounts_FromTree

from gcutils.gubbinsfuncs import annotate_HHR_with_GCE_overlaps

from gcutils.general import check_overlap_with_gene_group 

In [5]:
RE_CoordCols = ("seqname", "start_0based", "end_1based")
HmRegion_CoordCols = ("Chr", "Start", "End")


### Set matplotlib text export settings for Adobe Illustrator

In [6]:
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

#### Pandas Viewing Settings

In [7]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

# Import/parse processed H37rv genome annotations

In [8]:
RepoRef_Dir = "../../References"

AnnotatedGenes_And_IntergenicRegions_RepoRef_Dir = f"{RepoRef_Dir}/201027_H37rv_AnnotatedGenes_And_IntergenicRegions"
H37Rv_GenomeAnnotations_Genes_TSV = f"{AnnotatedGenes_And_IntergenicRegions_RepoRef_Dir}/H37Rv_GenomeAnnotations.Genes.tsv"

## H37Rv Gene Annotations TSV
H37Rv_GenomeAnno_Genes_DF = pd.read_csv(H37Rv_GenomeAnnotations_Genes_TSV, sep = "\t")
H37Rv_GeneInfo_Subset_DF = H37Rv_GenomeAnno_Genes_DF[["H37rv_GeneID", "Symbol", "Feature", "Functional_Category", "Is_Pseudogene", "Product", "PEandPPE_Subfamily", "ExcludedGroup_Category"]]

RvID_To_Symbol_Dict = dict(H37Rv_GeneInfo_Subset_DF[['H37rv_GeneID', 'Symbol']].values)

ESX_Genes_List_TSV = f"{RepoRef_Dir}/190927_H37rv_ListOf_ESXgenes.tsv"
Esx_Genes_DF = pd.read_csv(ESX_Genes_List_TSV, sep = '\t')

In [9]:
H37Rv_GenomeAnno_Genes_DF.head(1)

,Chrom,Start,End,Strand,H37rv_GeneID,Symbol,Feature,Functional_Category,Is_Pseudogene,Product,PEandPPE_Subfamily,ExcludedGroup_Category
0,NC_000962.3,0,1524,+,Rv0001,dnaA,CDS,information pathways,No,Chromosomal replication initiator protein DnaA,NaN,NotExcluded


### Define paths to H37Rv genome masking schemes

In [10]:
RepoRef_Dir = "../../References"

MaskingSchemes_Dir = f"{RepoRef_Dir}/Mtb_H37Rv_MaskingSchemes"
PLC_Scheme_BED = f"{MaskingSchemes_Dir}/201027_Mtb_H37rv_pLC_Regions_CoscollaExcludedGenes.bed"


## Define relevant gene lists for analysis (PE/PPE, Esx, 13E12 gene)

In [11]:
ListOf_Esx_Symbols = list(Esx_Genes_DF["symbol"].values)
ListOf_Esx_RvIDs = list(Esx_Genes_DF["gene_id"].values)

In [12]:
listOf_PEPPE_Symbols = list( H37Rv_GenomeAnno_Genes_DF.query(" Functional_Category == 'PE/PPE' ")["Symbol"].values )
listOf_PEPPE_RvIDs = list( H37Rv_GenomeAnno_Genes_DF.query(" Functional_Category == 'PE/PPE' ")["H37rv_GeneID"].values )

In [13]:
listOf_13E12_Region_RvIDs = ["Rv0094c", "Rv0095c", "Rv0393", "Rv1572c", "Rv1572c", "Rv1128c", "Rv1148c", "Rv1587c", "Rv1588c", "Rv1702c", "Rv1945", "Rv2100", "Rv3466", "Rv3467"]  


# Parse in homology H37Rv mapping results (k19w19)

In [14]:
Main_Project_Dir = "/n/data1/hms/dbmi/farhat/mm774/Projects/Mtb_WGA_Analysis_V8"

H37_Rv_MM2_HomologyMapping_Dir = f"{Main_Project_Dir}/220502.H37Rv.HomologyMapping.k19w19.ProcessedData"       

# Define paths to output TSVs

### Homologous regions (MERGED)
H37Rv_HomologyRegions_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/H37rv.HomologousRegions.k19w19.tsv"

### Homology map (pairwise alignments)
H37Rv_HomologyMap_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/H37rv.HomologyMap.k19w19.tsv"
H37Rv_HomologyMap_NoOverlap_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/H37rv.HomologyMap.k19w19.NoOverlap.tsv"
H37Rv_HomologyMap_OnlyOverlap_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/H37rv.HomologyMap.k19w19.OnlyOverlap.tsv"

H37Rv_HomologyMap_NoOverlap_Processed_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/H37rv.HmMap.k19w19.NoOverlap.Processed.V2.tsv"

### Variants from the homology map alignments
H37Rv_HmMap_Var_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/H37rv.HomologyMap.k19w19.variants.tsv"
H37Rv_HmMap_Var_SNPs_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/H37rv.HomologyMap.k19w19.variants.snps.tsv"


### Parse in labeled homology-regions (w/ unique IDs)

In [15]:
HM_MergedRegions_Anno_DF = pd.read_csv(H37Rv_HomologyRegions_TSV, sep="\t")
HM_MergedRegions_Anno_DF.head(3)

,HmRegion_Num,Chr,Start,End,Num_Ovrlap_Hm_Regions,Center,Length,num_HomologRegions_NonOvrlap,num_HomologRegions_NonOvrlap_MinSeqID99,num_HomologRegions_NonOvrlap_MinSeqID100,Overlap_Genes,Overlap_TE,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,HmRegionID
0,0,NC_000962.3,80184,80523,1,80353.5,339,1,1,1,Rv0071,0,False,False,False,True,HmRegion_000
1,1,NC_000962.3,80623,82664,1,81643.5,2041,1,1,1,"Rv0072,Rv0073",0,False,False,False,True,HmRegion_001
2,2,NC_000962.3,103705,105130,2,104417.5,1425,2,2,2,"Rv0094c,Rv0095c",0,False,False,True,False,HmRegion_002


### Parse in homology-map DFs (pairwise alignments between all homologous regions)

In [16]:

Mtb_HM_PAF_DF = pd.read_csv(H37Rv_HomologyMap_TSV, sep="\t")
Mtb_HM_PAF_DF_NoOverlapRegions = pd.read_csv(H37Rv_HomologyMap_NoOverlap_TSV, sep="\t")
Mtb_HM_PAF_DF_OnlyOverlapRegions = pd.read_csv(H37Rv_HomologyMap_OnlyOverlap_TSV, sep="\t")

HmPair_DF = pd.read_csv(H37Rv_HomologyMap_NoOverlap_Processed_TSV, sep="\t")

In [17]:
HmPair_DF.head()

,Query_Name,Query_Start,Query_End,Strand,Target_Name,Target_Start,Target_End,Num_ResidueMatches,Aln_BlockLength,Prop_Match,Target_Middle,Query_Middle,Dist_Middles,Query_Length,Target_Length,QueryOverlap_Genes,TargetOverlap_Genes,QueryOverlap_TE,TargetOverlap_TE,Query_Contains_Esx,Query_Contains_PEPPE,Query_Contains_REP13E12,Query_NoOverlap_Wi_PEPPE_Esx_13E12_Genes,SeqID,Dist,QueryCoords,TargetCoords,QueryToTarget_ID,Query_Cluster,Query_Cluster_Start,Query_Cluster_End,Target_Cluster,Target_Cluster_Start,Target_Cluster_End
0,NC_000962.3,80184,80523,+,NC_000962.3,3082465,3082769,283,340,0.832353,3082617.0,80353.5,3002263.5,339,304,Rv0071,Rv2774c,0,0,False,False,False,True,83.235294,1409268.5,NC_000962.3:80184-80523,NC_000962.3:3082465-3082769,NC_000962.3:80184-80523;NC_000962.3:3082465-30...,0,80184,80523,134,3082465,3082769
1,NC_000962.3,80623,82664,+,NC_000962.3,2882289,2884330,1650,2058,0.801749,2883309.5,81643.5,2801666.0,2041,2041,"Rv0072,Rv0073","Rv2563,glnQ",0,0,False,False,False,True,80.174927,1609866.0,NC_000962.3:80623-82664,NC_000962.3:2882289-2884330,NC_000962.3:80623-82664;NC_000962.3:2882289-28...,1,80623,82664,124,2882289,2884330
2,NC_000962.3,103705,105090,-,NC_000962.3,3883535,3884921,1346,1390,0.968345,3884228.0,104397.5,3779830.5,1385,1386,"Rv0094c,Rv0095c","Rv3466,Rv3467",0,0,False,False,True,False,96.834532,631701.5,NC_000962.3:103705-105090,NC_000962.3:3883535-3884921,NC_000962.3:103705-105090;NC_000962.3:3883535-...,2,103705,105130,177,3883535,3884921
3,NC_000962.3,103779,105130,+,NC_000962.3,1788513,1789865,1316,1354,0.971935,1789189.0,104454.5,1684734.5,1351,1352,"Rv0094c,Rv0095c","Rv1587c,Rv1588c",0,0,False,False,True,False,97.193501,1684734.5,NC_000962.3:103779-105130,NC_000962.3:1788513-1789865,NC_000962.3:103779-105130;NC_000962.3:1788513-...,2,103705,105130,73,1788513,1789865
4,NC_000962.3,149571,149808,-,NC_000962.3,1357376,1357613,195,241,0.809129,1357494.5,149689.5,1207805.0,237,237,PE_PGRS2,PE14,0,0,False,True,False,False,80.912863,1207805.0,NC_000962.3:149571-149808,NC_000962.3:1357376-1357613,NC_000962.3:149571-149808;NC_000962.3:1357376-...,3,149571,149808,55,1357376,1357613


### Parse in HomologyMap variants DFs

In [18]:
Mtb_HM_Var_DF = pd.read_csv(H37Rv_HmMap_Var_TSV, sep="\t")
Mtb_HM_Var_SNPs_DF = pd.read_csv(H37Rv_HmMap_Var_SNPs_TSV, sep="\t")


# Parse TGEN-1K SR-WGS sample metadata

In [19]:
Repo_MainDir = "../.."

Repo_DataDir = f"{Repo_MainDir}/Data"

Repo_RunInfoDir = f"{Repo_MainDir}/runInfo_TSVs"

TGen_1K_SampleInfo_CSV_PATH = f"{Repo_RunInfoDir}/TBportals.allTGEN_srrIds.forMax.csv"

TGen_1K_SM_V1_ResultsSummary_Dir = f"{Repo_DataDir}/Tgen1K_WGS_RunMetadata/211019_SM_TGen_1K_V1_ResultsSummary"

TGen_1K_SampleInfo_Filt_TSV_PATH = f"{Repo_DataDir}/Tgen1K_WGS_RunMetadata/211019_SM_TGen_1K_SampleInfo_V1.F2andCovFiltered.tsv"


TGen_1K_SRWGS_Stats_Filt_DF = pd.read_csv(TGen_1K_SampleInfo_Filt_TSV_PATH, sep = "\t")
TGen_1K_SRWGS_Stats_Filt_DF["PrimaryLineage"] = TGen_1K_SRWGS_Stats_Filt_DF["PrimaryLineage_Ill"]
TGen_1K_SRWGS_Stats_Filt_DF["SampleID"] = TGen_1K_SRWGS_Stats_Filt_DF["SampleName"]
TGen_1K_SRWGS_Stats_Filt_DF["Lineage"] = TGen_1K_SRWGS_Stats_Filt_DF["LineageCall_Illumina"]
TGen_1K_SRWGS_Stats_Filt_DF.shape

(937, 10)

### Define dictionaries that map sampleID to metadata labels

In [20]:
TGEN037_ID_To_PrimLineage_Dict = dict(TGen_1K_SRWGS_Stats_Filt_DF[['SampleID', 'PrimaryLineage']].values)
TGEN037_ID_To_SubLineage_Dict = dict( TGen_1K_SRWGS_Stats_Filt_DF[["SampleID", "Lineage"]].values)
TGEN037_ID_To_Dataset_Dict = dict(TGen_1K_SRWGS_Stats_Filt_DF[['SampleID', 'Dataset_Tag']].values)

# Define directory paths to `TGEN-1K` Gubbins results

In [21]:
# Define varaint calling pipeline output directories

TGen1K_SRWGS_OutputDir = "/n/data1/hms/dbmi/farhat/mm774/Projects/Mtb-GeneConv/250721.TGEN1K.GCAnalysis"


# Define path to TGEN01K Gubbins output (SR-WGS based predictions)
TGen1K_SRWGS_Gubbins_OutDir = f"{TGen1K_SRWGS_OutputDir}/Gubbins_Analysis_V1"

Gubbins_V1_OutputDir = TGen1K_SRWGS_Gubbins_OutDir

Gubbins_OutPrefix = "Tgen_937CI.Gubbins.FromSNVs.V1"

Gubbins_Prefix_PATH                = f"{TGen1K_SRWGS_Gubbins_OutDir}/{Gubbins_OutPrefix}"

Gubbins_NodeLabelledTree_PATH      = f"{Gubbins_Prefix_PATH}.node_labelled.final_tree.tre"

Gubbins_BranchStats_CSV_PATH       = f"{Gubbins_Prefix_PATH}.per_branch_statistics.csv"

Gubbins_RecombPreds_GFF            = f"{Gubbins_Prefix_PATH}.recombination_predictions.gff"
Gubbins_RecombPreds_EMBL           = f"{Gubbins_Prefix_PATH}.recombination_predictions.embl"

Gubbins_RecombPreds_RenamedChr_GFF = f"{Gubbins_Prefix_PATH}.recombination_predictions.RenamedCHR.gff"
Gubbins_RecombPreds_RenamedChr_BED = f"{Gubbins_Prefix_PATH}.recombination_predictions.RenamedCHR.bed"      

Gubbins_BaseReconstruction_EMBL    = f"{Gubbins_Prefix_PATH}.branch_base_reconstruction.embl"


# Begin Gubbins results processing

## 1) Parse Gubbins Tree w/ ETE3

In [22]:
i_Gubbins_T = Tree(Gubbins_NodeLabelledTree_PATH, format = 1)


## 2) Parsing over each node of the tree (ETE3), get ML BRANCH LENGTH and Number of descendents per node

In [23]:
Tree_BranchLen_Dict, NumDescendants_PerLeaf_Dict = get_BranchLengths_and_DescendantCounts_FromTree(i_Gubbins_T)

In [24]:
print( len(list(Tree_BranchLen_Dict.keys())) )

1872


In [25]:
print( len(list(NumDescendants_PerLeaf_Dict.keys())) )

1872


## 3) Label each LEAF within phylogeny by isolate's MTBC Lineage

In [26]:
count = 0
for n in i_Gubbins_T.get_leaves():
    n.add_feature("Primary_lineage", TGEN037_ID_To_PrimLineage_Dict.get(n.name, "Unknown Lineage") )
    n.add_feature("Sublineage",      TGEN037_ID_To_SubLineage_Dict.get(n.name, "Unknown Lineage") )
    count +=1
    
print(count)
    
i_Gubbins_T.sort_descendants(attr='Primary_lineage')


937


## 4) Infer lineage each node of the tree (ETE3)

In [27]:
node_To_PrimaryLin_Dict = {}

for node in i_Gubbins_T.iter_descendants("postorder"):
    
    listOf_ChildLineages = []
    
    for child_node in node.get_descendants():
        if child_node.is_leaf():
            listOf_ChildLineages.append(  (child_node.Primary_lineage) )
                #print(node.name, listOf_ChildLineages)
        
    set_Of_ChildLineages = list(set(listOf_ChildLineages))
    
    if len(set_Of_ChildLineages) == 1:
        OnlyOneLineage = True
    else:
        OnlyOneLineage = False
    
    if OnlyOneLineage:
        node_To_PrimaryLin_Dict[node.name] = set_Of_ChildLineages[0]

node_To_PrimaryLin_Dict.update(TGEN037_ID_To_PrimLineage_Dict)
    

In [28]:
child_node.name

'SRR10380011'

In [29]:
child_node.Primary_lineage

'lineage4'

## 5) Create a mapping of the primary lineage of each NODE to SampleID

### Output "node_To_PrimaryLin_Dict" dictionary 

In [30]:
Gubbins_NodeToPriLineage_Dict_JSON = f"{Gubbins_V1_OutputDir}/{Gubbins_OutPrefix}.NodeToPrimaryLineage.json"

with open(Gubbins_NodeToPriLineage_Dict_JSON, 'w') as json_file:
    json.dump(node_To_PrimaryLin_Dict, json_file)


#### test reading back in the JSON

In [31]:
with open(Gubbins_NodeToPriLineage_Dict_JSON) as json_file:
    node_To_PrimaryLin_Dict = json.load(json_file)

In [32]:
len(list(node_To_PrimaryLin_Dict.keys()))

1871

In [33]:
list(node_To_PrimaryLin_Dict.keys())[:5]

['internal_1702',
 'internal_1701',
 'internal_1704',
 'internal_1703',
 'internal_1700']

In [34]:
node_To_PrimaryLin_Dict["internal_1701"]

'lineage2'

In [35]:
#node_To_PrimaryLin_Dict

# Parse & Annotate Gubbins Recombination GFFs

### This gives us info at the individual event level (inferred by Gubbins)

In [36]:
Gubbins_Recomb_Events_DF = get_RecombEvents_From_Gubbins_GFF_V2(Gubbins_RecombPreds_RenamedChr_GFF)

Gubbins_Recomb_Events_DF["Lineage"] = Gubbins_Recomb_Events_DF["Child_Node"].map(node_To_PrimaryLin_Dict).fillna("None")

Gubbins_Recomb_Events_DF.shape

(27, 16)

In [37]:
#Gubbins_Recomb_Events_DF.query(" Child_Node == 'DNA086' ")

In [38]:
H37Rv_GenomeAnno_Genes_DF.head(3)

,Chrom,Start,End,Strand,H37rv_GeneID,Symbol,Feature,Functional_Category,Is_Pseudogene,Product,PEandPPE_Subfamily,ExcludedGroup_Category
0,NC_000962.3,0,1524,+,Rv0001,dnaA,CDS,information pathways,No,Chromosomal replication initiator protein DnaA,NaN,NotExcluded
1,NC_000962.3,2051,3260,+,Rv0002,dnaN,CDS,information pathways,No,DNA polymerase III (beta chain) DnaN (DNA nucl...,NaN,NotExcluded
2,NC_000962.3,3279,4437,+,Rv0003,recF,CDS,information pathways,No,DNA replication and repair protein RecF (singl...,NaN,NotExcluded


## Label Gubbins Recomb Events by overlapping genes

In [39]:
print(Gubbins_Recomb_Events_DF.shape)
Gubbins_Recomb_Events_DF = label_Gubbins_Events_DF_ByOvrLap_H37RvGenes(Gubbins_Recomb_Events_DF, 
                                                                       H37Rv_GenomeAnno_Genes_DF)
print(Gubbins_Recomb_Events_DF.shape)

(27, 16)
(27, 18)


In [40]:
Gubbins_Recomb_Events_DF.head()

,seqname,source,feature,start_1based,end_1based,score,strand,Parent_Node,Child_Node,neg_log_likelihood,snp_count,taxa_List,start_0based,CenterOfRegion,EventLen,Lineage,Overlap_Genes,Overlap_Gene_RvIDs
0,NC_000962.3,GUBBINS,CDS,3894773,3894791,0.000,.,internal_945,SRR10380054,307.551808,5,[SRR10380054],3894772,3894781.5,19,lineage4,PPE60,Rv3478
1,NC_000962.3,GUBBINS,CDS,3894773,3894791,0.000,.,internal_944,internal_945,516.670442,5,"[SRR10380054, SRR7516353]",3894772,3894781.5,19,lineage4,PPE60,Rv3478
2,NC_000962.3,GUBBINS,CDS,1340052,1341254,0.000,.,internal_953,SRR10380108,397.130081,5,[SRR10380108],1340051,1340652.5,1203,lineage4,"PPE18,esxK,esxL","Rv1196,Rv1197,Rv1198"
3,NC_000962.3,GUBBINS,CDS,3253265,3253297,0.000,.,internal_963,internal_964,499.956438,6,"[SRR10380210, SRR10380212]",3253264,3253280.5,33,lineage4,ppsB,Rv2932
4,NC_000962.3,GUBBINS,CDS,3766679,3767840,0.000,.,internal_963,internal_964,477.520633,7,"[SRR10380210, SRR10380212]",3766678,3767259.0,1162,lineage4,"PPE56,Rv3351c","Rv3350c,Rv3351c"


## Let's label each putative recombination event by whether it overlaps with one of the 3 gene-groups (Esx, PE/PPE, REP13E12 repeats, or None)

In [41]:
print("# of Esx genes:", len(ListOf_Esx_RvIDs))
print("# of PE/PPE genes:", len(listOf_PEPPE_RvIDs))
print("# of 13E12 Repeat Region genes:", len(listOf_13E12_Region_RvIDs))

# of Esx genes: 23
# of PE/PPE genes: 168
# of 13E12 Repeat Region genes: 14


In [42]:
# Apply the overlap function
G = Gubbins_Recomb_Events_DF.copy()
G["Contains_Esx"] = G["Overlap_Gene_RvIDs"].apply(lambda x: check_overlap_with_gene_group(x, ListOf_Esx_RvIDs))
G["Contains_PEPPE"] = G["Overlap_Gene_RvIDs"].apply(lambda x: check_overlap_with_gene_group(x, listOf_PEPPE_RvIDs))
G["Contains_REP13E12"] = G["Overlap_Gene_RvIDs"].apply(lambda x: check_overlap_with_gene_group(x, listOf_13E12_Region_RvIDs))
G["NoOverlap_Wi_PEPPE_Esx_13E12_Genes"] = ~(G["Contains_Esx"] | G["Contains_PEPPE"] | G["Contains_REP13E12"] )
Gubbins_Recomb_Events_DF = G.copy()

#### Peak at stats of overlap with gene groups commonly effected by gene conversion

In [43]:
Gubbins_Recomb_Events_DF["Contains_Esx"].value_counts()

Contains_Esx
False    25
True      2
Name: count, dtype: int64

In [44]:
Gubbins_Recomb_Events_DF["Contains_PEPPE"].value_counts()

Contains_PEPPE
True     17
False    10
Name: count, dtype: int64

In [45]:
Gubbins_Recomb_Events_DF["Contains_REP13E12"].value_counts()

Contains_REP13E12
False    24
True      3
Name: count, dtype: int64

In [46]:
Gubbins_Recomb_Events_DF["NoOverlap_Wi_PEPPE_Esx_13E12_Genes"].value_counts()

NoOverlap_Wi_PEPPE_Esx_13E12_Genes
False    20
True      7
Name: count, dtype: int64

## Generate EventIDs for all detected events

In [47]:
## 1) Create ordered version of all putative recombination events.
## - Let's sort by the following columns: ["start_1based", "end_1based", "Parent_Node", "Child_Node", "Lineage", "Overlap_Genes"]

Gub_ColumnsToSortBy = ["start_1based", "end_1based", "Parent_Node", "Child_Node", "Lineage", "Overlap_Genes"]

Gubbins_Recomb_Events_DF = Gubbins_Recomb_Events_DF.sort_values(Gub_ColumnsToSortBy, kind="mergesort").reset_index(drop=True)

Gubbins_Recomb_Events_DF["EventNum"] = Gubbins_Recomb_Events_DF.index + 1
Gubbins_Recomb_Events_DF["EventID"] = "Event_" + Gubbins_Recomb_Events_DF["EventNum"].astype(str).str.rjust(3, '0')

Gubbins_Recomb_Events_DF = Gubbins_Recomb_Events_DF.drop("EventNum", axis = 1)


In [48]:
Gubbins_Recomb_Events_DF["EventLen"].describe()

count      27.000000
mean      291.814815
std       375.186419
min         8.000000
25%        27.000000
50%       105.000000
75%       434.500000
max      1203.000000
Name: EventLen, dtype: float64

In [49]:
Gubbins_Recomb_Events_DF["neg_log_likelihood"].describe()

count      27.000000
mean      497.292412
std       372.486732
min        69.803852
25%       307.969255
50%       419.038754
75%       562.462215
max      1430.476419
Name: neg_log_likelihood, dtype: float64

In [50]:
Gubbins_Recomb_Events_DF.shape

(27, 23)

In [51]:
Gubbins_Recomb_Events_DF.head(2)

,seqname,source,feature,start_1based,end_1based,score,strand,Parent_Node,Child_Node,neg_log_likelihood,snp_count,taxa_List,start_0based,CenterOfRegion,EventLen,Lineage,Overlap_Genes,Overlap_Gene_RvIDs,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,EventID
0,NC_000962.3,GUBBINS,CDS,333120,333212,0.000,.,internal_985,internal_986,158.013591,8,"[SRR10379945, SRR7516364, SRR10380193, SRR1037...",333119,333165.5,93,lineage4,"vapC25,vapB25","Rv0277c,Rv0277A",False,False,False,True,Event_001
1,NC_000962.3,GUBBINS,CDS,473800,473971,0.000,.,internal_1262,SRR6807674,70.598206,6,[SRR6807674],473799,473885.0,172,lineage4,Rv0393,Rv0393,False,False,True,False,Event_002


### Annotate each event by whether an HHR overlaps

In [52]:
RE_CoordCols = ("seqname", "start_0based", "end_1based")
HmAlnPAF_CoordCols = ("Query_Name", "Query_Start", "Query_End")
HHR_CoordCols = ("Chr", "Start", "End")

#### Label NucDiv DF by whether a paralogous aligmment pair overlaps
Gubbins_Recomb_Events_DF = bf.count_overlaps(Gubbins_Recomb_Events_DF,
                                             Mtb_HM_PAF_DF, 
                                             cols1 = RE_CoordCols, cols2 = HmAlnPAF_CoordCols )

Gubbins_Recomb_Events_DF.rename(columns={'count': 'HmAln_Count'}, inplace=True)
Gubbins_Recomb_Events_DF["HmAln_Ovrlap"] = np.where(Gubbins_Recomb_Events_DF['HmAln_Count'] > 0, 1, 0)



#### Label NucDiv DF by whether a paralogous region overlaps
Gubbins_Recomb_Events_DF = bf.count_overlaps(Gubbins_Recomb_Events_DF,
                                             HM_MergedRegions_Anno_DF,
                                             cols1 = RE_CoordCols, cols2 = HHR_CoordCols )

Gubbins_Recomb_Events_DF.rename(columns={'count': 'HHR_Count'}, inplace=True)
Gubbins_Recomb_Events_DF["HHR_Ovrlap"] = np.where(Gubbins_Recomb_Events_DF['HHR_Count'] > 0, 1, 0)



In [53]:
Gubbins_Recomb_Events_DF.shape

(27, 27)

In [54]:
Gubbins_Recomb_Events_DF.head()

,seqname,source,feature,start_1based,end_1based,score,strand,Parent_Node,Child_Node,neg_log_likelihood,snp_count,taxa_List,start_0based,CenterOfRegion,EventLen,Lineage,Overlap_Genes,Overlap_Gene_RvIDs,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,EventID,HmAln_Count,HmAln_Ovrlap,HHR_Count,HHR_Ovrlap
0,NC_000962.3,GUBBINS,CDS,333120,333212,0.000,.,internal_985,internal_986,158.013591,8,"[SRR10379945, SRR7516364, SRR10380193, SRR1037...",333119,333165.5,93,lineage4,"vapC25,vapB25","Rv0277c,Rv0277A",False,False,False,True,Event_001,1,1,1,1
1,NC_000962.3,GUBBINS,CDS,473800,473971,0.000,.,internal_1262,SRR6807674,70.598206,6,[SRR6807674],473799,473885.0,172,lineage4,Rv0393,Rv0393,False,False,True,False,Event_002,1,1,1,1
2,NC_000962.3,GUBBINS,CDS,841087,841448,0.000,.,internal_1061,internal_1062,1023.196321,14,"[SRR10380227, SRR10379958]",841086,841267.0,362,lineage4,"vapB31,vapC31","Rv0748,Rv0749",False,False,False,True,Event_003,1,1,1,1
3,NC_000962.3,GUBBINS,CDS,842026,842065,0.000,.,internal_941,SRR6807728,1309.669305,9,[SRR6807728],842025,842045.0,40,lineage4,Rv0750,Rv0750,False,False,False,True,Event_004,1,1,1,1
4,NC_000962.3,GUBBINS,CDS,842051,842111,0.000,.,internal_1177,internal_1181,374.835261,7,"[SRR6807675, SRR10379983, SRR6807700, SRR68076...",842050,842080.5,61,lineage4,Rv0750,Rv0750,False,False,False,True,Event_005,1,1,1,1


### Annotate each event by the overlapping HHRs (Each HHR ID)

In [56]:
Gubbins_Recomb_Events_DF = label_Gubbins_Events_DF_ByOvrLap_HHRs(Gubbins_Recomb_Events_DF, HM_MergedRegions_Anno_DF)

In [57]:
HM_MergedRegions_Anno_DF.shape

(197, 17)

In [58]:
Gubbins_Recomb_Events_DF.shape

(27, 28)

In [59]:
Gubbins_Recomb_Events_DF.shape

(27, 28)

In [60]:
Gubbins_Recomb_Events_DF.query(" HHR_Ovrlap == 0 ")["Overlap_HHRs"].values


array([''], dtype=object)

In [61]:
#Gubbins_Recomb_Events_DF["Overlap_HHRs"].value_counts()

## Output updated Gubbins_Recomb_Events_DF to TSV

In [62]:
Gubbins_RecombPreds_Anno_TSV = f"{Gubbins_V1_OutputDir}/{Gubbins_OutPrefix}.recombination_predictions.Anno.tsv"

Gubbins_Recomb_Events_DF.to_csv(Gubbins_RecombPreds_Anno_TSV, sep="\t", index=False)


### Explore a bit the detected GC events in TGEN dataset

In [63]:
Gubbins_Recomb_Events_DF.head()

,seqname,source,feature,start_1based,end_1based,score,strand,Parent_Node,Child_Node,neg_log_likelihood,snp_count,taxa_List,start_0based,CenterOfRegion,EventLen,Lineage,Overlap_Genes,Overlap_Gene_RvIDs,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,EventID,HmAln_Count,HmAln_Ovrlap,HHR_Count,HHR_Ovrlap,Overlap_HHRs
0,NC_000962.3,GUBBINS,CDS,333120,333212,0.000,.,internal_985,internal_986,158.013591,8,"[SRR10379945, SRR7516364, SRR10380193, SRR1037...",333119,333165.5,93,lineage4,"vapC25,vapB25","Rv0277c,Rv0277A",False,False,False,True,Event_001,1,1,1,1,HmRegion_009
1,NC_000962.3,GUBBINS,CDS,473800,473971,0.000,.,internal_1262,SRR6807674,70.598206,6,[SRR6807674],473799,473885.0,172,lineage4,Rv0393,Rv0393,False,False,True,False,Event_002,1,1,1,1,HmRegion_018
2,NC_000962.3,GUBBINS,CDS,841087,841448,0.000,.,internal_1061,internal_1062,1023.196321,14,"[SRR10380227, SRR10379958]",841086,841267.0,362,lineage4,"vapB31,vapC31","Rv0748,Rv0749",False,False,False,True,Event_003,1,1,1,1,HmRegion_031
3,NC_000962.3,GUBBINS,CDS,842026,842065,0.000,.,internal_941,SRR6807728,1309.669305,9,[SRR6807728],842025,842045.0,40,lineage4,Rv0750,Rv0750,False,False,False,True,Event_004,1,1,1,1,HmRegion_032
4,NC_000962.3,GUBBINS,CDS,842051,842111,0.000,.,internal_1177,internal_1181,374.835261,7,"[SRR6807675, SRR10379983, SRR6807700, SRR68076...",842050,842080.5,61,lineage4,Rv0750,Rv0750,False,False,False,True,Event_005,1,1,1,1,HmRegion_032


In [64]:
Gubbins_Recomb_Events_DF.query(" HHR_Ovrlap == 0 ")


,seqname,source,feature,start_1based,end_1based,score,strand,Parent_Node,Child_Node,neg_log_likelihood,snp_count,taxa_List,start_0based,CenterOfRegion,EventLen,Lineage,Overlap_Genes,Overlap_Gene_RvIDs,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,EventID,HmAln_Count,HmAln_Ovrlap,HHR_Count,HHR_Ovrlap,Overlap_HHRs
20,NC_000962.3,GUBBINS,CDS,3477266,3477370,0.000,.,internal_1082,SRR10397163,175.415705,5,[SRR10397163],3477265,3477317.5,105,lineage4,Rv3108,Rv3108,False,False,False,True,Event_021,0,0,0,0,


# Count recomb events per region

### Count inferred recombination events (Gubbins) per: <br> 
a) 1 kb region <br> 
b) gene <br> 
c) merged-homologous region <br>

## Read in annotated windows of the H37Rv genome

In [65]:
RepoRef_Dir = "../../References"

H37Rv_Windows_Dir = f"{RepoRef_Dir}/H37Rv_GenomeWindows"

H37Rv_1kb_Win_Anno_TSV = f"{H37Rv_Windows_Dir}/H37Rv.1000bp.Windows.Anno.tsv"

Rv_1kb_Win_DF = pd.read_csv(H37Rv_1kb_Win_Anno_TSV, sep = "\t")
Rv_1kb_Window_Start_To_Genes_Dict = dict(Rv_1kb_Win_DF[['Start', 'Overlap_Genes']].values)


### a) Let's count events per 1 kb window

In [66]:
RvWin_CoordCols = ("Chrom", "Start", "End")
RE_CoordCols = ("seqname", "start_0based", "end_1based")

Rv_1kb_RE_Count_DF = bf.count_overlaps(Rv_1kb_Win_DF,
                                       Gubbins_Recomb_Events_DF,
                                       cols1 = RvWin_CoordCols,
                                       cols2 = RE_CoordCols)

Rv_1kb_RE_Count_DF.rename(columns={'count': 'pGCE_Count'}, inplace=True)

Rv_1kb_RE_Count_DF["CenterOfRegion"] = ((Rv_1kb_RE_Count_DF["Start"] + Rv_1kb_RE_Count_DF["End"]) / 2)

Rv_1kb_RE_Count_DF.shape

(4412, 7)

In [67]:
Rv_1kb_RE_Count_DF.sort_values("pGCE_Count", ascending=False).head(4)

,Chrom,Start,End,Overlap_Genes,Middle,pGCE_Count,CenterOfRegion
1340,NC_000962.3,1340000,1341000,"PPE18,esxK",1340500.0,4,1340500.0
3894,NC_000962.3,3894000,3895000,"PE31,PPE60",3894500.0,3,3894500.0
3377,NC_000962.3,3377000,3378000,PPE46,3377500.0,3,3377500.0
1339,NC_000962.3,1339000,1340000,"PE13,PPE18",1339500.0,2,1339500.0


### d) Let's count events per gene

In [68]:
RE_CoordCols = ("seqname", "start_0based", "end_1based")
GenomeAnno_CoordCols = ("Chrom", "Start", "End")

Rv_Genes_RE_Count_DF = bf.count_overlaps(H37Rv_GenomeAnno_Genes_DF,
                                         Gubbins_Recomb_Events_DF,
                                         cols1 = GenomeAnno_CoordCols,
                                         cols2 = RE_CoordCols)

Rv_Genes_RE_Count_DF.rename(columns={'count': 'pGCE_Count'}, inplace=True)
Rv_Genes_RE_Count_DF["CenterOfRegion"] = ((Rv_Genes_RE_Count_DF["Start"] + Rv_Genes_RE_Count_DF["End"]) / 2)

Rv_Genes_RE_Count_DF.shape

(4079, 14)

In [69]:
#Rv_Genes_RE_Count_DF.sort_values("pGCE_Count", ascending=False).head(4)

### e) Count events per HHR (MERGED homologous region)

In [70]:
RE_CoordCols = ("seqname", "start_0based", "end_1based")
HmRegion_CoordCols = ("Chr", "Start", "End")

Rv_HHRs_RE_Count_DF = bf.count_overlaps(HM_MergedRegions_Anno_DF,
                                        Gubbins_Recomb_Events_DF,
                                        cols1 = HmRegion_CoordCols,
                                        cols2 = RE_CoordCols)

Rv_HHRs_RE_Count_DF["CenterOfRegion"] = ((Rv_HHRs_RE_Count_DF["Start"] + Rv_HHRs_RE_Count_DF["End"]) / 2)

Rv_HHRs_RE_Count_DF.rename(columns={'count': 'pGCE_Count'}, inplace=True)

#### Annotate HHRs by the eventIDs that overlap
Rv_HHRs_RE_Count_DF = annotate_HHR_with_GCE_overlaps(Rv_HHRs_RE_Count_DF,
                                                     Gubbins_Recomb_Events_DF)


Rv_HHRs_RE_Count_DF.shape

(197, 20)

In [71]:
Rv_HHRs_RE_Count_DF[Rv_HHRs_RE_Count_DF['HmRegionID'].isna()]

,HmRegion_Num,Chr,Start,End,Num_Ovrlap_Hm_Regions,Center,Length,num_HomologRegions_NonOvrlap,num_HomologRegions_NonOvrlap_MinSeqID99,num_HomologRegions_NonOvrlap_MinSeqID100,Overlap_Genes,Overlap_TE,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,HmRegionID,pGCE_Count,CenterOfRegion,Overlap_GC_EventIDs


In [72]:
Rv_HHRs_RE_Count_DF.head(2)

,HmRegion_Num,Chr,Start,End,Num_Ovrlap_Hm_Regions,Center,Length,num_HomologRegions_NonOvrlap,num_HomologRegions_NonOvrlap_MinSeqID99,num_HomologRegions_NonOvrlap_MinSeqID100,Overlap_Genes,Overlap_TE,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,HmRegionID,pGCE_Count,CenterOfRegion,Overlap_GC_EventIDs
0,0,NC_000962.3,80184,80523,1,80353.5,339,1,1,1,Rv0071,0,False,False,False,True,HmRegion_000,0,80353.5,None
1,1,NC_000962.3,80623,82664,1,81643.5,2041,1,1,1,"Rv0072,Rv0073",0,False,False,False,True,HmRegion_001,0,81643.5,None


In [73]:
Rv_HHRs_RE_Count_DF.sort_values("pGCE_Count", ascending=False).head(1)

,HmRegion_Num,Chr,Start,End,Num_Ovrlap_Hm_Regions,Center,Length,num_HomologRegions_NonOvrlap,num_HomologRegions_NonOvrlap_MinSeqID99,num_HomologRegions_NonOvrlap_MinSeqID100,Overlap_Genes,Overlap_TE,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,HmRegionID,pGCE_Count,CenterOfRegion,Overlap_GC_EventIDs
52,52,NC_000962.3,1338923,1340433,2,1339678.0,1510,2,2,2,"PE13,PPE18",0,False,True,False,False,HmRegion_052,5,1339678.0,"Event_008,Event_009,Event_010,Event_011,Event_012"


### Output pGCE counts per region TSVs

In [74]:
Gubbins_EventsPer_1kb_H37Rv_TSV_PATH = f"{Gubbins_V1_OutputDir}/{Gubbins_OutPrefix}.H37Rv.EventsPer1kb.tsv"

Gubbins_EventsPer_Gene_H37Rv_TSV_PATH = f"{Gubbins_V1_OutputDir}/{Gubbins_OutPrefix}.H37Rv.EventsPerGene.tsv"
Gubbins_EventsPer_MgHomologousRegion_H37Rv_TSV_PATH = f"{Gubbins_V1_OutputDir}/{Gubbins_OutPrefix}.H37Rv.EventsPerMergedHomologousRegion.tsv"

Rv_1kb_RE_Count_DF.to_csv(Gubbins_EventsPer_1kb_H37Rv_TSV_PATH, sep="\t", index=False)
Rv_Genes_RE_Count_DF.to_csv(Gubbins_EventsPer_Gene_H37Rv_TSV_PATH, sep="\t", index=False)
Rv_HHRs_RE_Count_DF.to_csv(Gubbins_EventsPer_MgHomologousRegion_H37Rv_TSV_PATH, sep="\t", index=False)


In [75]:
#!ls -alh $Gubbins_V1_OutputDir

# Parse base reconstruction of Gubbins (EMBL)

In [76]:
Gubbins_SNPs_All_DF = parse_BaseReconstruction_EMBL_Gubbins(Gubbins_BaseReconstruction_EMBL)
Gubbins_SNPs_All_DF.shape

(37125, 6)

In [77]:
Gubbins_SNPs_All_DF.head()

,Pos_1based,Parent_Node,Child_Node,Parent_Call,Child_Call,taxa_List
0,158366,internal_945,SRR10380054,A,G,SRR10380054
1,251267,internal_945,SRR10380054,T,C,SRR10380054
2,258256,internal_945,SRR10380054,A,G,SRR10380054
3,354351,internal_945,SRR10380054,A,G,SRR10380054
4,408382,internal_945,SRR10380054,T,C,SRR10380054


In [78]:
Gubbins_SNPs_All_DF.shape

(37125, 6)

# Label all SNP mutational events by associated EventID

In [79]:
Gubbins_SNPs_All_Anno_DF = annotate_Gubbins_SNP_Events_By_RecombEventID(Gubbins_SNPs_All_DF, Gubbins_Recomb_Events_DF)
Gubbins_SNPs_All_Anno_DF.shape

(37125, 7)

In [80]:
Gubbins_SNPs_EventOnly_Anno_DF = Gubbins_SNPs_All_Anno_DF.query("EventID != 'None'")
Gubbins_SNPs_EventOnly_Anno_DF.shape

(177, 7)

In [81]:
Gubbins_SNPs_All_Anno_DF["EventID"].value_counts().head(10)

EventID
None         36948
Event_003       14
Event_004        9
Event_019        8
Event_001        8
Event_009        8
Event_024        8
Event_018        8
Event_012        7
Event_011        7
Name: count, dtype: int64

In [82]:
Gubbins_SNPs_All_Anno_DF.head(3)

,Pos_1based,Parent_Node,Child_Node,Parent_Call,Child_Call,taxa_List,EventID
0,1439598,internal_1000,SRR7516342,A,C,SRR7516342,None
1,2423237,internal_1000,SRR7516342,A,G,SRR7516342,None
2,4401400,internal_1000,SRR7516342,A,C,SRR7516342,None


In [83]:
Gubbins_SNPs_All_Anno_DF.tail(3)

,Pos_1based,Parent_Node,Child_Node,Parent_Call,Child_Call,taxa_List,EventID
37122,4313128,internal_ROOT,internal_939,T,C,SRR10379936 SRR6807728 SRR10397269 SRR75...,None
37123,4329782,internal_ROOT,internal_939,A,G,SRR10379936 SRR6807728 SRR10397269 SRR75...,None
37124,4408923,internal_ROOT,internal_939,T,C,SRR10379936 SRR6807728 SRR10397269 SRR75...,None


In [84]:
Gubbins_SNPs_EventOnly_Anno_DF.shape

(177, 7)

In [85]:
Gubbins_SNPs_All_Anno_DF.shape

(37125, 7)

### How many total SNPs associated with ALL EVENTS detected?

Answer: Agreement between recombination-events DF and SNPs Df

In [86]:
Gubbins_Recomb_Events_DF["snp_count"].sum()

177

In [87]:
Gubbins_SNPs_EventOnly_Anno_DF.shape

(177, 7)

## Output Gubbins' Base-Reconstruction SNP dataframe (Annotated)

In [88]:
Gubbins_BaseReconstruction_Anno_TSV = f"{Gubbins_V1_OutputDir}/{Gubbins_OutPrefix}.branch_base_reconstruction.AnnoByEvent.All.tsv"   
Gubbins_BaseReconstruction_Anno_EventOnly_TSV = f"{Gubbins_V1_OutputDir}/{Gubbins_OutPrefix}.branch_base_reconstruction.AnnoByEvent.EventSNPsOnly.tsv"   


In [89]:
Gubbins_SNPs_All_Anno_DF.to_csv(Gubbins_BaseReconstruction_Anno_TSV, sep="\t", index=False)

In [90]:
Gubbins_SNPs_EventOnly_Anno_DF.to_csv(Gubbins_BaseReconstruction_Anno_EventOnly_TSV, sep="\t", index=False)

## TEST PARSING of Gubbins ASR SNP files

In [91]:
Gubbins_SNPs_All_Anno_DF = pd.read_csv(Gubbins_BaseReconstruction_Anno_TSV, sep = "\t")
Gubbins_SNPs_All_Anno_DF.shape

(37125, 7)

In [92]:
Gubbins_SNPs_EventOnly_Anno_DF = pd.read_csv(Gubbins_BaseReconstruction_Anno_EventOnly_TSV, sep = "\t")
Gubbins_SNPs_EventOnly_Anno_DF.shape

(177, 7)

In [93]:
Gubbins_SNPs_All_Anno_DF.head(5)

,Pos_1based,Parent_Node,Child_Node,Parent_Call,Child_Call,taxa_List,EventID
0,1439598,internal_1000,SRR7516342,A,C,SRR7516342,NaN
1,2423237,internal_1000,SRR7516342,A,G,SRR7516342,NaN
2,4401400,internal_1000,SRR7516342,A,C,SRR7516342,NaN
3,1644148,internal_1000,SRR7516365,T,C,SRR7516365,NaN
4,9615,internal_1001,SRR10380027,A,T,SRR10380027,NaN


In [94]:
Gubbins_SNPs_EventOnly_Anno_DF.head(3)

,Pos_1based,Parent_Node,Child_Node,Parent_Call,Child_Call,taxa_List,EventID
0,3765797,internal_1022,internal_1023,G,C,SRR10380179 SRR10380112 SRR10379961 SRR1038...,Event_022
1,3765808,internal_1022,internal_1023,A,C,SRR10380179 SRR10380112 SRR10379961 SRR1038...,Event_022
2,3765809,internal_1022,internal_1023,C,G,SRR10380179 SRR10380112 SRR10379961 SRR1038...,Event_022


# Process the Gubbins' Branch stats

In [95]:
G_BranchStats_DF = pd.read_csv(Gubbins_BranchStats_CSV_PATH, sep = "\t")
G_BranchStats_DF.shape

(1873, 10)

In [96]:
G_BranchStats_DF.head()

,Node,Total SNPs,Num of SNPs inside recombinations,Num of SNPs outside recombinations,Num of Recombination Blocks,Bases in Recombinations,r/m,rho/theta,Genome Length,Bases in Clonal Frame
0,SRR10397218,2,0,2,0,566,0.0,0.0,4411526,4410958
1,SRR10397230,84,0,84,0,506,0.0,0.0,4411521,4411014
2,SRR7516345,82,0,82,0,506,0.0,0.0,4411519,4411012
3,SRR10379936,396,0,396,0,0,0.0,0.0,4411478,4411478
4,SRR7516429,51,0,51,0,506,0.0,0.0,4411497,4410990


In [97]:
G_BranchStats_DF.sort_values("Num of Recombination Blocks", ascending=False).head(10)

,Node,Total SNPs,Num of SNPs inside recombinations,Num of SNPs outside recombinations,Num of Recombination Blocks,Bases in Recombinations,r/m,rho/theta,Genome Length,Bases in Clonal Frame
954,internal_964,102,13,89,2,1460,0.146067,0.022472,4411448,4411180
16,SRR6807728,289,17,272,2,353,0.062500,0.007353,4411524,4411524
1022,internal_1039,150,6,144,1,609,0.041667,0.006944,4411528,4411528
964,internal_968,315,8,307,1,629,0.026059,0.003257,4411498,4411498
166,SRR10397175,19,7,12,1,779,0.583333,0.083333,4411531,4411531
718,SRR10430374,63,4,59,1,7,0.067797,0.016949,4411474,4411474
995,internal_995,67,6,61,1,112,0.098361,0.016393,4411508,4411415
1127,internal_1066,103,5,98,1,58,0.051020,0.010204,4411532,4411532
958,internal_946,14,5,9,1,267,0.555556,0.111111,4411474,4411474
860,SRR10397163,34,5,29,1,162,0.172414,0.034483,4411515,4411456


## Annotated Branch stats by lineage

In [98]:
G_BranchStats_DF["Lineage"] = G_BranchStats_DF["Node"].map(node_To_PrimaryLin_Dict).fillna("None")

G_BranchStats_DF["BranchLen"] = G_BranchStats_DF["Node"].map(Tree_BranchLen_Dict).fillna("None")

G_BranchStats_DF["Num_Tips_Downstream"] = G_BranchStats_DF["Node"].map(NumDescendants_PerLeaf_Dict).fillna("None")

G_BranchStats_DF["Total_SNPs"] = G_BranchStats_DF["Total SNPs"]


### Remove last branch for unrooted tree

In [99]:
G_BranchStats_DF.tail(2)

,Node,Total SNPs,Num of SNPs inside recombinations,Num of SNPs outside recombinations,Num of Recombination Blocks,Bases in Recombinations,r/m,rho/theta,Genome Length,Bases in Clonal Frame,Lineage,BranchLen,Num_Tips_Downstream,Total_SNPs
1871,internal_1325,454,0,454,0,0,0.0,0.0,4411532,4411532,None,0.0,550.0,454
1872,internal_ROOT,0,0,0,0,0,0.0,0.0,4411532,4411532,None,None,None,0


In [100]:
# Remove the lasts root node, it has no actual branch length

G_BranchStats_Filt_DF = G_BranchStats_DF.query("BranchLen != 'None'")


In [101]:
G_BranchStats_Filt_DF.shape[0]

1872

In [102]:
G_BranchStats_DF.shape[0]

1873

### Output updated branch stats TSV

In [103]:
G_BranchStats_WithLineage_CSV = f"{Gubbins_V1_OutputDir}/{Gubbins_OutPrefix}.per_branch_statistics.WithLineagePerNode.tsv"

G_BranchStats_Filt_DF.to_csv(G_BranchStats_WithLineage_CSV, sep="\t", index=False)


In [104]:
!ls -lah $G_BranchStats_WithLineage_CSV

-rw-rw-r-- 1 mm774 hpc_farhat 134K Aug  3 15:50 /n/data1/hms/dbmi/farhat/mm774/Projects/Mtb-GeneConv/250721.TGEN1K.GCAnalysis/Gubbins_Analysis_V1/Tgen_937CI.Gubbins.FromSNVs.V1.per_branch_statistics.WithLineagePerNode.tsv


In [105]:
G_BranchStats_Filt_DF.head(4)

,Node,Total SNPs,Num of SNPs inside recombinations,Num of SNPs outside recombinations,Num of Recombination Blocks,Bases in Recombinations,r/m,rho/theta,Genome Length,Bases in Clonal Frame,Lineage,BranchLen,Num_Tips_Downstream,Total_SNPs
0,SRR10397218,2,0,2,0,566,0.0,0.0,4411526,4410958,lineage4,2.001308,0.0,2
1,SRR10397230,84,0,84,0,506,0.0,0.0,4411521,4411014,lineage4,70.174469,0.0,84
2,SRR7516345,82,0,82,0,506,0.0,0.0,4411519,4411012,lineage4,74.72773,0.0,82
3,SRR10379936,396,0,396,0,0,0.0,0.0,4411478,4411478,lineage4,322.007782,0.0,396


In [106]:
G_BranchStats_Filt_DF.tail(4)

,Node,Total SNPs,Num of SNPs inside recombinations,Num of SNPs outside recombinations,Num of Recombination Blocks,Bases in Recombinations,r/m,rho/theta,Genome Length,Bases in Clonal Frame,Lineage,BranchLen,Num_Tips_Downstream,Total_SNPs
1868,internal_1332,176,0,176,0,0,0.0,0.0,4411532,4411532,lineage2,20.880798,543.0,176
1869,internal_1328,24,0,24,0,0,0.0,0.0,4411532,4411532,lineage2,78.000427,547.0,24
1870,internal_1327,77,0,77,0,0,0.0,0.0,4411532,4411532,lineage2,336.556,548.0,77
1871,internal_1325,454,0,454,0,0,0.0,0.0,4411532,4411532,None,0.0,550.0,454
